<a href="https://colab.research.google.com/github/Dahan15/projects/blob/main/forecasting_gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
arashnic_learn_time_series_forecasting_from_gold_price_path = kagglehub.dataset_download('arashnic/learn-time-series-forecasting-from-gold-price')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Load the dataset
data = pd.read_csv("/kaggle/input/learn-time-series-forecasting-from-gold-price/gold_price_data.csv")  # Adjust the path if needed
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

# Preprocess the data



In [ ]:
import tensorflow as tf
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

size_of_sampel=int(len(data)*0.8)
train=scaled_data[0:size_of_sampel]
test=scaled_data[size_of_sampel:]
window_size = 20
X, y = [], []
for i in range(window_size, len(scaled_data)):
    X.append(scaled_data[i-window_size:i])
    y.append(scaled_data[i])

X = np.array(X)
y = np.array(y)

# Train/val split
train_size = int(len(X) * 0.8)
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

# Dataset using tf.data
batch_size = 128
buffer_size = 1000

train_data = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_data = train_data.cache().shuffle(buffer_size).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

model = Sequential()
model.add(SimpleRNN(64, activation='relu', return_sequences=True, input_shape=(window_size, 1)))
model.add(SimpleRNN(32, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

# Fit model

In [ ]:
# Train the model
model.fit(train_data, epochs=200, verbose=2)

In [ ]:
X, y = [], []
for i in range(window_size, len(test)):
    X.append(test[i-window_size:i])
    y.append(test[i])

X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
def evaluate_model(model, X_val, y_val):
    forecast_val = model.predict(X_val)
    mae_val = mean_absolute_error(y_val, forecast_val)
    mse_val = mean_squared_error(y_val, forecast_val)
    return mae_val, mse_val

def plot_predictions(forecast_val, y_val, title="Baseline model plot"):
    plt.figure(figsize=(10, 6))
    plt.plot(forecast_val, label='Predicted Validation')
    plt.plot(y_val, label='Actual Validation')
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
# plot
forecast = model.predict(X)
plot_predictions(forecast, y, title="LSTM Model Performance")
# evaluate
mae, mse = evaluate_model(model, X, y)
print(f'MAE: {mae}, MSE: {mse}')